# Pandas paso a paso: ventas de una tienda

En este ejemplo trabajaremos con **10 ventas ficticias**. Queremos responder preguntas concretas: ¿qué se vendió?, ¿hay datos incompletos?, ¿cuánto ingresó cada venta? y ¿qué categoría ingresó más?

Una **tabla** en pandas se llama `DataFrame`: tiene filas (ventas) y columnas (datos de cada venta). Una sola columna, como `producto`, es una `Series`.

Ejecuta las celdas de arriba abajo. Los datos están dentro del notebook, así que no necesitas descargar archivos.

## 1. Preparar el entorno

Instala `pandas` para trabajar con tablas y `matplotlib` para dibujar el gráfico final. `%pip` instala los paquetes en el entorno del kernel de este notebook. Si se instalan por primera vez, reinicia el kernel y vuelve a ejecutar las celdas.

In [ ]:
%pip install pandas matplotlib

## 2. Cargar y mirar la tabla

Los datos de abajo tienen formato **CSV**: la primera línea nombra las columnas y cada línea siguiente es una venta. `StringIO` permite leer ese texto como si fuera un archivo; así podemos practicar `pd.read_csv()` sin depender de una ruta en tu ordenador.

`import pandas as pd` carga el módulo con el alias `pd`, por eso usamos `pd.read_csv(...)`. `from io import StringIO` trae un nombre concreto, que usamos directamente como `StringIO(...)`.

Al final de la celda se muestra el `DataFrame` completo. Fíjate en que hay **10 filas** y que algunas celdas de `vendedor` están vacías.

In [ ]:
import pandas as pd  # Cargamos pandas con el nombre corto "pd".
from io import StringIO  # Traemos StringIO para leer texto CSV como si fuera un archivo.
from IPython.display import display  # Traemos display para ver tablas bien formateadas en el notebook.
import matplotlib.pyplot as plt  # Cargamos el módulo de gráficos con el nombre corto "plt".

# Datos ficticios: cada línea después de la cabecera representa una venta.
datos_csv = """fecha,producto,categoria,unidades,precio_unitario,vendedor
2026-10-01,Cuaderno,Papelería,2,3.50,Ana
2026-10-01,Bolígrafo,Papelería,5,1.20,Luis
2026-10-02,Taza,Hogar,1,8.00,Ana
2026-10-02,Cuaderno,Papelería,3,3.50,
2026-10-03,Lámpara,Hogar,1,15.00,Luis
2026-10-03,Bolígrafo,Papelería,4,1.20,Ana
2026-10-04,Taza,Hogar,2,8.00,Luis
2026-10-04,Cuaderno,Papelería,1,3.50,Ana
2026-10-05,Lámpara,Hogar,2,15.00,
2026-10-05,Bolígrafo,Papelería,3,1.20,Luis"""

ventas = pd.read_csv(StringIO(datos_csv))  # Convertimos el texto CSV en un DataFrame.
print(f"La tabla tiene {ventas.shape[0]} filas y {ventas.shape[1]} columnas.")  # Mostramos su tamaño.
display(ventas)  # Mostramos las ventas originales antes de modificarlas.

## 3. Inspeccionar antes de calcular

`head(3)` enseña las tres primeras filas; `dtypes` indica qué tipo de dato guarda cada columna. `isna().sum()` cuenta los valores ausentes por columna. Un valor ausente aparece como `NaN` en la tabla.

En este caso falta el nombre del vendedor en dos ventas. **No inventaremos un vendedor**: las etiquetaremos como «Sin asignar». Los números de esas ventas siguen siendo válidos.

In [ ]:
display(ventas.head(3))  # Vemos una muestra pequeña de la tabla.
print("Tipos de columna:")  # Anunciamos qué información aparece debajo.
print(ventas.dtypes)  # Comprobamos qué columnas son texto y cuáles son números.
print()  # Dejamos una línea en blanco entre resultados.
print("Valores ausentes por columna:")  # Anunciamos la comprobación de celdas vacías.
print(ventas.isna().sum())  # Contamos las celdas vacías en cada columna.

## 4. Preparar los datos

Rellenamos solo los nombres de vendedor ausentes con «Sin asignar». Convertimos `fecha` de texto a fecha real de pandas para poder trabajar con días y meses más adelante. Después añadimos una columna calculada: `importe = unidades × precio_unitario`.

Por ejemplo, la primera venta tiene 2 cuadernos a 3,50 €: su importe es **7,00 €**.

In [ ]:
ventas["vendedor"] = ventas["vendedor"].fillna("Sin asignar")  # Etiquetamos los vendedores desconocidos.
ventas["fecha"] = pd.to_datetime(ventas["fecha"])  # Convertimos el texto de fecha a tipo fecha.
ventas["importe"] = ventas["unidades"] * ventas["precio_unitario"]  # Calculamos el ingreso de cada venta.

display(ventas[["fecha", "producto", "unidades", "precio_unitario", "importe", "vendedor"]])  # Vemos las columnas más útiles.
print(f"Ingresos totales: {ventas['importe'].sum():.2f} €")  # Sumamos los importes de todas las ventas.

## 5. Seleccionar y filtrar

`ventas["producto"]` selecciona una sola columna (`Series`). Para quedarnos solo con algunas **filas y columnas**, usamos `.loc[filas, columnas]`. Aquí la condición `ventas["categoria"] == "Hogar"` selecciona las ventas de Hogar.

Comprueba que el resultado contiene solo tazas y lámparas.

In [ ]:
es_hogar = ventas["categoria"] == "Hogar"  # Creamos una condición verdadera para las ventas de Hogar.
ventas_hogar = ventas.loc[es_hogar, ["fecha", "producto", "unidades", "importe"]]  # Elegimos esas filas y cuatro columnas.
display(ventas_hogar)  # Mostramos la tabla filtrada.

## 6. Resumir por categoría

`groupby("categoria")` reúne las ventas de la misma categoría. Después, `sum()` suma sus importes. Así pasamos de diez filas de ventas a **dos filas de resumen**.

`as_index=False` mantiene `categoria` como columna normal. El resultado responde: **¿cuánto ingresó cada categoría?** Con estos datos, esperamos **69,00 € en Hogar** y **35,40 € en Papelería**; juntos suman **104,40 €**.

In [ ]:
resumen = ventas.groupby("categoria", as_index=False)["importe"].sum()  # Sumamos el importe dentro de cada categoría.
resumen = resumen.sort_values("importe", ascending=False)  # Ordenamos de mayor a menor ingreso.
display(resumen)  # Mostramos el resumen numérico.

## 7. Ver el resumen en un gráfico

Cada barra representa una categoría; su altura es el importe total en euros. El gráfico y la tabla anterior muestran la misma información de dos maneras. Según estos datos ficticios, **Hogar ingresó más que Papelería**.

In [ ]:
ax = resumen.plot.bar(x="categoria", y="importe", color=["#2563eb", "#f59e0b"], legend=False, figsize=(7, 4))  # Dibujamos una barra por categoría.
ax.set_title("Ingresos por categoría")  # Añadimos un título claro.
ax.set_xlabel("Categoría")  # Nombramos el eje horizontal.
ax.set_ylabel("Ingresos (€)")  # Nombramos el eje vertical.
ax.tick_params(axis="x", rotation=0)  # Dejamos los nombres horizontales para leerlos mejor.
plt.tight_layout()  # Ajustamos los márgenes para que no se corte texto.
plt.show()  # Mostramos el gráfico dentro del notebook.

## Prueba tú

1. Cambia la condición del filtro para mostrar solo **Papelería**.
2. Calcula cuántas **unidades** se vendieron en total con `ventas["unidades"].sum()`.
3. Cambia `groupby("categoria")` por `groupby("producto")` y observa qué producto ingresó más.

Si cambias datos o código, vuelve a ejecutar las celdas desde la sección correspondiente.